# Chapter 3 sc/snRNA-seq figures (PDEs & mechanosensitive proteins)

This notebook is a *scaffold* to generate the figures your PI requested (Fig 1–8).  
It assumes you have an `.h5ad` file with AnnData object `adata`.

**Your dataset summary (from your note):**
- shape: (881081, 33234)
- `adata.X`: CSR sparse
- `adata.obs` key columns: `cell_type`, `Region_x`, `Primary.Genetic.Diagnosis`, `Sample`, `donor_id`, `disease` (check)
- `adata.var['feature_name']`: gene symbols

---

In [ ]:
eval "$(~/miniforge3/bin/conda shell.bash hook)"
conda create -n scrnajupyter python=3.12 ipykernel jupyter_client
conda activate scrnajupyter
conda install numpy pandas anndata scanpy
python -m ipykernel install --user --name python312_scrnajupyter --display-name "Python3.12 (scrnajupyter)"

In [2]:
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp 
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
print(ad.__version__)
import os
import re
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120)

/var/tmp/pbs.1898552.pbs-7/ipykernel_1504990/3288260042.py:8: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(ad.__version__)


0.12.7


## 0) Load data

Put your `.h5ad` in the same folder or change `H5AD_PATH`.


In [3]:
adata = ad.read_h5ad("local.h5ad")

In [4]:
print("shape:", adata.shape)
print("X type:", type(adata.X), "sparse:", sp.issparse(adata.X))
print("obs cols:", list(adata.obs.columns)[:40])
print("var cols:", list(adata.var.columns)[:10])

shape: (881081, 33234)
X type: <class 'scipy.sparse._csr.csr_matrix'> sparse: True
obs cols: ['Sample', 'donor_id', 'Region_x', 'Primary.Genetic.Diagnosis', 'n_genes', 'n_counts', 'percent_mito', 'percent_ribo', 'scrublet_score_z', 'scrublet_score_log', 'solo_score', 'cell_states', 'Assigned', 'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'sex_ontology_term_id', 'assay_ontology_term_id', 'organism_ontology_term_id', 'is_primary_data', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'suspension_type', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage']
var cols: ['feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype']


## 1) Quick QC sanity checks + key categories

This prints unique values so you can decide the exact filters for:
- donor vs DCM
- whole heart vs left ventricle (LV)


In [5]:
def show_top_values(col, n=20):
    if col not in adata.obs.columns:
        print(f"[missing] {col}")
        return
    vc = adata.obs[col].value_counts()
    print(f"\n== {col} ==")
    print(vc.head(n))
    print("n_unique:", vc.shape[0])

for c in ["Primary.Genetic.Diagnosis", "disease", "Region_x", "cell_type", "Sample", "donor_id"]:
    show_top_values(c, n=20)


== Primary.Genetic.Diagnosis ==
Primary.Genetic.Diagnosis
control    282372
LMNA       122995
TTN        114925
PKP2        89240
RBM20       76558
PVneg       66323
DES         36692
TNNC1       30631
PLN         25552
TPM1        11632
BAG3         7444
TNNT2        5696
FKTN         4053
DSP          3793
FLNC         3175
Name: count, dtype: int64
n_unique: 15

== disease ==
disease
dilated cardiomyopathy                             482581
normal                                             282372
arrhythmogenic right ventricular cardiomyopathy    104496
non-compaction cardiomyopathy                       11632
Name: count, dtype: int64
n_unique: 4

== Region_x ==
Region_x
LV    650358
RV    230723
Name: count, dtype: int64
n_unique: 2

== cell_type ==
cell_type
cardiac muscle cell             311418
mural cell                      170281
fibroblast of cardiac tissue    142816
endothelial cell                115548
myeloid cell                     57036
native cell                 

## 2) UMAP panels for the presentation (optional but recommended)

If `X_umap` exists (it does in your printout), this is quick and clean.

In [17]:
# UMAP colored by key annotations
sc.settings.figdir = "./figures"   # 存图文件夹
sc.settings.set_figure_params(dpi=300)

cols = ["cell_type", "Primary.Genetic.Diagnosis", "disease", "Region_x", "sex"]

for c in cols:
    sc.pl.umap(
        adata,
        color=c,
        frameon=False,
        show=False,
        save=f"_UMAP_{c}.png"
    )

In [ ]:
# Metadata bar plots to accompany the UMAP panels
sc.settings.figdir = "./figures"
os.makedirs(sc.settings.figdir, exist_ok=True)

import matplotlib.pyplot as plt
import pandas as pd


def metadata_barplot(adata, col, fname, title=None, mode="cells", rotate=45):
    if col not in adata.obs.columns:
        print(f"[skip] {col} not found")
        return

    df = adata.obs[[col]].copy()
    df[col] = df[col].astype(str).fillna("NA")

    if mode == "donors":
        if "donor_id" not in adata.obs.columns:
            print(f"[skip] donor_id not found, cannot do donor-level barplot for {col}")
            return
        df = adata.obs[["donor_id", col]].copy()
        df[col] = df[col].astype(str).fillna("NA")
        counts = (
            df.drop_duplicates(["donor_id", col])
              .groupby(col)["donor_id"]
              .nunique()
              .sort_values(ascending=False)
        )
        ylabel = "Number of donors"
    else:
        counts = df[col].value_counts(dropna=False)
        ylabel = "Number of cells"

    fig_w = max(5, 0.45 * len(counts) + 2)
    plt.figure(figsize=(fig_w, 4.5))
    ax = counts.plot(kind="bar")
    plt.title(title or f"{col} ({mode})")
    plt.ylabel(ylabel)
    plt.xlabel("")
    plt.xticks(rotation=rotate, ha="right")

    for p in ax.patches:
        h = p.get_height()
        ax.annotate(f"{int(h):,}",
                    (p.get_x() + p.get_width()/2., h),
                    ha='center', va='bottom', fontsize=8,
                    xytext=(0, 3), textcoords='offset points')

    plt.tight_layout()
    plt.savefig(os.path.join(sc.settings.figdir, f"{fname}.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

# Suggested set matching the UMAP annotations + sex
metadata_barplot(adata, "Primary.Genetic.Diagnosis", "Bar_Diagnosis_donors", title="Diagnosis distribution across donors", mode="donors", rotate=45)
metadata_barplot(adata, "disease", "Bar_Disease_donors", title="Disease distribution across donors", mode="donors", rotate=20)
metadata_barplot(adata, "Region_x", "Bar_Region_cells", title="Region distribution across cells", mode="cells", rotate=0)
metadata_barplot(adata, "cell_type", "Bar_CellType_cells", title="Cell-type distribution across cells", mode="cells", rotate=45)
metadata_barplot(adata, "sex", "Bar_Sex_donors", title="Sex distribution across donors", mode="donors", rotate=0)



## 3) Helper: get gene symbols and build PDE list

We use `adata.var['feature_name']` as gene symbol by default.

In [7]:
# Choose the gene symbol column
GENE_SYMBOL_COL = "feature_name" if "feature_name" in adata.var.columns else None
if GENE_SYMBOL_COL is None:
    raise ValueError("No obvious gene symbol column found. Check adata.var.columns.")

gene_symbols = adata.var[GENE_SYMBOL_COL].astype(str).values

# PDE genes: starts with PDE (human), plus a few known aliases if you want to extend later
pde_mask = np.array([bool(re.match(r"^PDE\d+[A-Z]?$", g)) for g in gene_symbols])
pde_genes = sorted(pd.unique(gene_symbols[pde_mask]).tolist())

print("n PDE genes detected:", len(pde_genes))
print("first 30:", pde_genes[:30])

n PDE genes detected: 24
first 30: ['PDE10A', 'PDE12', 'PDE1A', 'PDE1B', 'PDE1C', 'PDE2A', 'PDE3A', 'PDE3B', 'PDE4A', 'PDE4B', 'PDE4C', 'PDE4D', 'PDE5A', 'PDE6A', 'PDE6B', 'PDE6C', 'PDE6D', 'PDE6G', 'PDE6H', 'PDE7A', 'PDE7B', 'PDE8A', 'PDE8B', 'PDE9A']


In [8]:
def sort_pde_genes(pde_list):
    def key_func(g):
        # 提取 PDE 后面的数字
        m = re.match(r"PDE(\d+)([A-Z]?)", g)
        if m:
            return (int(m.group(1)), m.group(2))
        else:
            return (999, g)
    return sorted(pde_list, key=key_func)

pde_genes_sorted = sort_pde_genes(pde_genes)

print(pde_genes_sorted)


['PDE1A', 'PDE1B', 'PDE1C', 'PDE2A', 'PDE3A', 'PDE3B', 'PDE4A', 'PDE4B', 'PDE4C', 'PDE4D', 'PDE5A', 'PDE6A', 'PDE6B', 'PDE6C', 'PDE6D', 'PDE6G', 'PDE6H', 'PDE7A', 'PDE7B', 'PDE8A', 'PDE8B', 'PDE9A', 'PDE10A', 'PDE12']


In [9]:
pde_genes_heart = [g for g in pde_genes_sorted if not g.startswith("PDE6")]
print(pde_genes_heart)

['PDE1A', 'PDE1B', 'PDE1C', 'PDE2A', 'PDE3A', 'PDE3B', 'PDE4A', 'PDE4B', 'PDE4C', 'PDE4D', 'PDE5A', 'PDE7A', 'PDE7B', 'PDE8A', 'PDE8B', 'PDE9A', 'PDE10A', 'PDE12']


## 6) Mechanosensitive proteins (Fig 2 & 4)

The 50-gene mechanosensitive list is in an Excel file.

In [10]:
MEC_EXCEL = "Mechanosensitive_Protein_list_updated.xlsx"
mec_df = pd.read_excel(MEC_EXCEL)

mec_genes = mec_df["Gene Name"].dropna().astype(str).unique().tolist()
print("n mechanosensitive genes:", len(mec_genes))
print(mec_genes[:10])


n mechanosensitive genes: 50
['ACTC1', 'TTN', 'JPH2', 'VCL', 'CSRP3', 'ACTN2', 'MYOZ2', 'TCAP', 'ANKRD1', 'ANK1']


In [11]:
all_symbols = set(adata.var[GENE_SYMBOL_COL].astype(str).values)
missing = [g for g in mec_genes if g not in all_symbols]
present = [g for g in mec_genes if g in all_symbols]

print("present:", len(present))
print("missing:", len(missing))
print("missing genes:", missing)


present: 50
missing: 0
missing genes: []


## 4) Define filters: donor vs DCM, whole vs LV vs RV

**Default logic (edit if needed):**
- donor heart = `Primary.Genetic.Diagnosis == 'control'`
- DCM heart = everything else (excluding control)
- LV = `Region_x == 'LV'`
- RV = `Region_x == 'RV'`

This cell creates convenience AnnData subsets for whole heart, LV and RV figures.


In [12]:
COL_DIAG = "Primary.Genetic.Diagnosis"
COL_REGION = "Region_x"

mask_donor = adata.obs[COL_DIAG].astype(str).eq("control")
mask_dcm   = ~mask_donor

mask_lv = adata.obs[COL_REGION].astype(str).eq("LV")
mask_rv = adata.obs[COL_REGION].astype(str).eq("RV")

print("donor total:", int(mask_donor.sum()))
print("dcm total:", int(mask_dcm.sum()))
print("donor LV:", int((mask_donor & mask_lv).sum()))
print("donor RV:", int((mask_donor & mask_rv).sum()))
print("dcm LV:", int((mask_dcm & mask_lv).sum()))
print("dcm RV:", int((mask_dcm & mask_rv).sum()))

donor total: 282372
dcm total: 598709
donor LV: 219149
donor RV: 63223
dcm LV: 431209
dcm RV: 167500


## 5) Fig 1 & 3: PDE expression across cell types (whole, LV & RV)

For cross-cell-type display, **dotplot** usually looks better than dense heatmaps.
- groups: `cell_type`
- genes: all PDEs
- panels: whole heart, LV, RV

This keeps the format consistent across region-specific plots.


In [13]:
COL_DIAG   = "Primary.Genetic.Diagnosis"
COL_REGION = "Region_x"
GROUP_COL  = "cell_type"
GENE_SYMBOL_COL = "feature_name"   # 你数据里就是它

mask_donor = adata.obs[COL_DIAG].astype(str).eq("control")
mask_dcm   = ~mask_donor
mask_lv    = adata.obs[COL_REGION].astype(str).eq("LV")
mask_rv    = adata.obs[COL_REGION].astype(str).eq("RV")


### Note on memory usage
For the DCM whole-heart plots, use the memory-safe `dotplot_save()` below.
It subsets to the requested genes first and only then copies the reduced matrix, which avoids crashing when plotting the ~600k-cell DCM subset.



In [14]:
preferred_order = [
    "cardiac muscle cell",
    "endothelial cell",
    "fibroblast of cardiac tissue",
    "mural cell",
    "myeloid cell",
    "lymphocyte",
    "native cell",
    "cardiac neuron",
    "mast cell",
    "fat cell"
]
# 只保留存在的
preferred_order = [x for x in preferred_order if x in adata.obs[GROUP_COL].unique().tolist()]

adata.obs[GROUP_COL] = pd.Categorical(adata.obs[GROUP_COL], categories=preferred_order, ordered=True)


In [24]:
sc.settings.figdir = "./figures" # 你想存哪就改哪
os.makedirs(sc.settings.figdir, exist_ok=True)

def dotplot_save(ad_view, genes, title, fname,
                 dot_min=0.1, dot_max=0.8, smallest_dot=20,
                 standard_scale="var", dendrogram=False):
    # Exclude 'native cell' (annotated as unknown/unreliable in manuscript) from Fig1-4 plots
    if GROUP_COL in ad_view.obs.columns:
        _keep = ad_view.obs[GROUP_COL].astype(str).ne("native cell")
        if _keep.sum() < ad_view.n_obs:
            ad_view = ad_view[_keep].copy()
            # drop unused categories to avoid empty legend entries
            ad_view.obs[GROUP_COL] = ad_view.obs[GROUP_COL].astype("category").cat.remove_unused_categories()

    # 只保留真实存在的 genes（防报错）
    all_symbols = ad_view.var[GENE_SYMBOL_COL].astype(str).values
    symbol_set = set(all_symbols)
    genes_present = [g for g in genes if g in symbol_set]

    print(f"{fname}: genes present {len(genes_present)}/{len(genes)}")

    sc.pl.dotplot(
        ad_view,
        var_names=genes_present,          # ✅ 关键：用传入的 genes
        groupby=GROUP_COL,
        gene_symbols=GENE_SYMBOL_COL,
        standard_scale=standard_scale,
        dendrogram=dendrogram,         # =False先别聚类，保证 cell_type 顺序稳定
        dot_min=dot_min,
        dot_max=dot_max,
        smallest_dot=smallest_dot,   # 这个很重要，让小点别太小
        title=title,
        show=False,
        save=f"_{fname}.png"
    )


In [18]:
# Fig 1a donor whole heart
dotplot_save(
    adata[mask_donor],
    pde_genes_heart,
    "Fig 1a | Donor | PDEs across all cell types | Whole heart",
    "Fig1a_Donor_whole_PDE"
)

# Fig 1b donor LV
dotplot_save(
    adata[mask_donor & mask_lv],
    pde_genes_heart,
    "Fig 1b | Donor | PDEs across all cell types | LV",
    "Fig1b_Donor_LV_PDE"
)

# Fig 1c donor RV
dotplot_save(
    adata[mask_donor & mask_rv],
    pde_genes_heart,
    "Fig 1c | Donor | PDEs across all cell types | RV",
    "Fig1c_Donor_RV_PDE"
)

# Fig 3a DCM whole heart
dotplot_save(
    adata[mask_dcm],
    pde_genes_heart,
    "Fig 3a | DCM | PDEs across all cell types | Whole heart",
    "Fig3a_DCM_whole_PDE"
)

# Fig 3b DCM LV
dotplot_save(
    adata[mask_dcm & mask_lv],
    pde_genes_heart,
    "Fig 3b | DCM | PDEs across all cell types | LV",
    "Fig3b_DCM_LV_PDE"
)

# Fig 3c DCM RV
dotplot_save(
    adata[mask_dcm & mask_rv],
    pde_genes_heart,
    "Fig 3c | DCM | PDEs across all cell types | RV",
    "Fig3c_DCM_RV_PDE"
)


Fig1a_Donor_whole_PDE: genes present 18/18
Fig1b_Donor_LV_PDE: genes present 18/18
Fig1c_Donor_RV_PDE: genes present 18/18
Fig3a_DCM_whole_PDE: genes present 18/18
Fig3b_DCM_LV_PDE: genes present 18/18
Fig3c_DCM_RV_PDE: genes present 18/18


## 6) Mechanosensitive proteins (Fig 2 & 4)

You said the 50-gene mechanosensitive list is in an Excel file.

Put it here and name the column containing gene symbols (e.g. `gene_symbol`).

In [19]:
# Fig 2a: Donor whole heart
dotplot_save(
    adata[mask_donor],
    mec_genes,
    "Fig 2a | Donor | Mechanosensitive proteins across all cell types | Whole heart",
    "Fig2a_Donor_whole_Mech"
)

# Fig 2b: Donor LV
dotplot_save(
    adata[mask_donor & mask_lv],
    mec_genes,
    "Fig 2b | Donor | Mechanosensitive proteins across all cell types | LV",
    "Fig2b_Donor_LV_Mech"
)

# Fig 2c: Donor RV
dotplot_save(
    adata[mask_donor & mask_rv],
    mec_genes,
    "Fig 2c | Donor | Mechanosensitive proteins across all cell types | RV",
    "Fig2c_Donor_RV_Mech"
)

# Fig 4a: DCM whole heart
dotplot_save(
    adata[mask_dcm],
    mec_genes,
    "Fig 4a | DCM | Mechanosensitive proteins across all cell types | Whole heart",
    "Fig4a_DCM_whole_Mech"
)

# Fig 4b: DCM LV
dotplot_save(
    adata[mask_dcm & mask_lv],
    mec_genes,
    "Fig 4b | DCM | Mechanosensitive proteins across all cell types | LV",
    "Fig4b_DCM_LV_Mech"
)

# Fig 4c: DCM RV
dotplot_save(
    adata[mask_dcm & mask_rv],
    mec_genes,
    "Fig 4c | DCM | Mechanosensitive proteins across all cell types | RV",
    "Fig4c_DCM_RV_Mech"
)


Fig2a_Donor_whole_Mech: genes present 50/50
Fig2b_Donor_LV_Mech: genes present 50/50
Fig2c_Donor_RV_Mech: genes present 50/50
Fig4a_DCM_whole_Mech: genes present 50/50
Fig4b_DCM_LV_Mech: genes present 50/50
Fig4c_DCM_RV_Mech: genes present 50/50


## 7) Region-specific dot plots (Fig 5)

Subset a chosen cell type, then group by `Condition + Region` so the output format matches the earlier dot plots.

This gives you the same visual grammar as Fig 1–4, but now the x-axis groups become:
- Donor_LV
- Donor_RV
- DCM_LV
- DCM_RV


In [20]:
COL_DIAG   = "Primary.Genetic.Diagnosis"
COL_REGION = "Region_x"
GROUP_COL  = "cell_type"

# 1) condition: donor vs DCM
adata.obs["Condition"] = pd.Series(
    ["Donor" if x == "control" else "DCM" for x in adata.obs[COL_DIAG].astype(str)],
    index=adata.obs_names
)

# 2) 拼接 condition + region
adata.obs["CondRegion"] = adata.obs["Condition"].astype(str) + "_" + adata.obs[COL_REGION].astype(str)

# 3) 固定显示顺序：Donor_LV, Donor_RV, DCM_LV, DCM_RV
condregion_order = ["Donor_LV", "Donor_RV", "DCM_LV", "DCM_RV"]
adata.obs["CondRegion"] = pd.Categorical(adata.obs["CondRegion"], categories=condregion_order, ordered=True)

# sanity check
print(adata.obs["CondRegion"].value_counts(dropna=False))

CondRegion
DCM_LV      431209
Donor_LV    219149
DCM_RV      167500
Donor_RV     63223
Name: count, dtype: int64


In [40]:
#重新定义dotplot_save函数
sc.settings.figdir = "./figures"
os.makedirs(sc.settings.figdir, exist_ok=True)

def dotplot_save(ad_view, genes, title, fname,
                 groupby_col,
                 dot_min=0.05, dot_max=0.9, smallest_dot=20,
                 standard_scale="var", dendrogram=False):
    # 只保留真实存在的 genes
    all_symbols = ad_view.var[GENE_SYMBOL_COL].astype(str).values
    symbol_set = set(all_symbols)
    genes_present = [g for g in genes if g in symbol_set]
    print(f"{fname}: genes present {len(genes_present)}/{len(genes)}")

    # 关键：把表达矩阵从 view 切成可用（scanpy会自己处理稀疏）
    sc.pl.dotplot(
        ad_view,
        var_names=genes_present,
        groupby=groupby_col,          # ✅ Fig5 要用 CondRegion
        gene_symbols=GENE_SYMBOL_COL,
        standard_scale=standard_scale, # ✅ Fig5 我建议先关掉，颜色更“真实”
        dendrogram=dendrogram,
        dot_min=dot_min,
        dot_max=dot_max,
        smallest_dot=smallest_dot,
        cmap="Reds",                  # ✅ 给颜色一个明显的 colormap（不然容易发白）
        title=title,
        show=False,
        save=f"_{fname}.png"
    )


In [41]:
def subsample_by_group(adata_in, group_col, n_per_group=20000, seed=0):
    rng = np.random.default_rng(seed)
    idx_keep = []
    groups = adata_in.obs[group_col].astype(str)
    for g in adata_in.obs[group_col].cat.categories if hasattr(adata_in.obs[group_col], "cat") else groups.unique():
        m = np.where(groups.values == g)[0]
        if len(m) == 0:
            continue
        if len(m) > n_per_group:
            m = rng.choice(m, size=n_per_group, replace=False)
        idx_keep.append(m)
    idx_keep = np.concatenate(idx_keep) if len(idx_keep) else np.array([], dtype=int)
    return adata_in[idx_keep]  # view，不 copy

# CM view
CM_LABEL = "cardiac muscle cell"
mask_cm = adata.obs["cell_type"].astype(str).eq(CM_LABEL)
cm_view = adata[mask_cm]

# optional subsampling: each CondRegion at most 20k cells
cm_small = subsample_by_group(cm_view, "CondRegion", n_per_group=20000, seed=1)

# Fig 5a | CM | PDEs
dotplot_save(
    cm_small,
    pde_genes_heart,
    "Fig 5a | Cardiomyocytes | PDEs across LV/RV in Donor vs DCM",
    "Fig5a_CM_PDE_CondRegion",
    groupby_col="CondRegion",
    smallest_dot=25
)

# Fig 5b | CM | mechanosensitive genes
# Uncomment if you also want the mechanosensitive panel in the same format
# dotplot_save(
#     cm_small,
#     mec_genes,
#     "Fig 5b | Cardiomyocytes | Mechanosensitive proteins across LV/RV in Donor vs DCM",
#     "Fig5b_CM_Mech_CondRegion",
#     groupby_col="CondRegion",
#     smallest_dot=25
# )


Fig5a_CM_PDE_CondRegion: genes present 18/18


/rds/general/user/sl2221/home/miniforge3/envs/scrnajupyter/lib/python3.12/site-packages/anndata/_core/anndata.py:1190: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


In [34]:
# Optional: same Fig 5 style for endothelial cells
EC_LABEL = "endothelial cell"
mask_ec = adata.obs["cell_type"].astype(str).eq(EC_LABEL)
ec_view = adata[mask_ec]
ec_small = subsample_by_group(ec_view, "CondRegion", n_per_group=20000, seed=1)

dotplot_save(
    ec_small,
    pde_genes_heart,
    "Fig 5 optional | Endothelial cells | PDEs across LV/RV in Donor vs DCM",
    "Fig5_optional_EC_PDE_CondRegion",
    groupby_col="CondRegion",
    smallest_dot=25
)


Fig5_optional_EC_PDE_CondRegion: genes present 18/18


/rds/general/user/sl2221/home/miniforge3/envs/scrnajupyter/lib/python3.12/site-packages/anndata/_core/anndata.py:1190: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


In [35]:
PDE4_genes = [g for g in pde_genes_heart if g.startswith("PDE4")]

dotplot_save(
    cm_small,
    PDE4_genes,
    "Fig 5a. PDE4 expression by disease-region",
    "Fig5a_CM_PDE4_CondRegion",
    groupby_col="CondRegion"
)


Fig5a_CM_PDE4_CondRegion: genes present 4/4
